In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = None


# ============================================================
# Input/output paths: read summary CSVs only
# ============================================================

DATASET_ID = "Replogle_RPE"
PERTURBED_GROUP = "single"

EVAL_ROOT = Path(
    f"Perturbation/evaluation/{DATASET_ID}_pseudo_pairing_evaluation/{PERTURBED_GROUP}"
)

TASK_MODEL_ROOT = EVAL_ROOT / "downstream_mlp_task_models"
FORWARD_SUMMARY_CSV = TASK_MODEL_ROOT / "combined_forward_mlp_run_summary.csv"
INVERSE_SUMMARY_CSV = TASK_MODEL_ROOT / "combined_inverse_mlp_run_summary.csv"

OUT_DIR = TASK_MODEL_ROOT / "_summary_strategy_scatterplots_like_visualize"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# What to plot
# ============================================================

# Keep the old Visualize_metrics.ipynb plot layout:
# x-axis = model, color/dodge = pseudo-control strategy, one figure per metric.
MODELS_ORDER = ["mlp_expr", "scvi", "sccello", "scimilarity", "scgpt"]

# Use restored post-perturbation expression errors by default.
# To plot perturbation-effect delta errors instead, set this to "delta".
FORWARD_ERROR_MODE = "restored_expression"  # "restored_expression" or "delta"

METRICS_TO_PLOT = [
    "forward_effect_mae",
    "forward_effect_mse",
    "inverse_accuracy",
    "inverse_macro_f1",
    "inverse_macro_precision",
    "inverse_macro_recall",
    "inverse_macro_auc",
]

# Exact selected variant order, using the same __ slug format as your old notebook.
STRATEGY_ORDER = [
    "S0_naive_mean_control_reference",
    "S1_random_single_control__seed_000",
    "S2_random_average_controls__k_100__seed_000",
    "S3_SEACell_metacell_average__nmc_350__k_10__seed_000",
    "S4_SEACell_balanced_random_sample__nmc_500__seed_000",
    "S5_SEACell_OT_sampled_average__nmc_350__topk_05__seed_000",
    "S5_SEACell_OT_sampled_average__nmc_500__topk_05__seed_000",
]


# ============================================================
# Plot style copied from your Visualize_metrics.ipynb
# ============================================================

matplotlib.rcParams["svg.fonttype"] = "none"
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
matplotlib.rcParams["axes.linewidth"] = 0.8
matplotlib.rcParams["xtick.major.width"] = 0.8
matplotlib.rcParams["ytick.major.width"] = 0.8

STRATEGY_PLOT_LABELS = {
    "S0_naive_mean_control_reference": "Naive\nmean\ncontrol",
    "S1_random_single_control": "Random\nsingle\ncontrol",
    "S2_random_average_controls": "Random\naverage\ncontrol",
    "S4_SEACell_balanced_random_sample": "Metacell\nbalanced\nrandom",
    "S3_SEACell_metacell_average": "Random\nmetacell\naverage",
    "S5_SEACell_OT_sampled_average": "Metacell OT\nsampled\naverage",
}

STRATEGY_BASE_COLORS = {
    "S0_naive_mean_control_reference": "#4D4D4D",
    "S1_random_single_control": "#6E8FB2",
    "S2_random_average_controls": "#7DA494",
    "S3_SEACell_metacell_average": "#E5A79A",
    "S4_SEACell_balanced_random_sample": "#EAB67A",
    "S5_SEACell_OT_sampled_average": "#9F8DB8",
}

S5_VARIANT_COLORS = {
    "200&5": "#D49AB5",
    "350&5": "#7D5284",
    "500&5": "#B66699",
}

DODGE_WIDTH = 0.42
POINT_SIZE = 72
ALPHA = 0.9
SAVE_PNG = True
SAVE_PDF = True
SAVE_SVG = True
DPI = 300
FIGSIZE = (10, 6)
DISPLAY_FIGURES = True


# ============================================================
# Metric aliases from summary CSVs to old notebook-style names
# ============================================================

if FORWARD_ERROR_MODE == "delta":
    FORWARD_MAE_CANDIDATES = ["strategy_specific_delta_mae", "forward_effect_mae", "perturbation_effect_mae"]
    FORWARD_MSE_CANDIDATES = ["strategy_specific_delta_mse", "forward_effect_mse", "perturbation_effect_mse"]
    FORWARD_LABEL_PREFIX = "Forward effect"
else:
    FORWARD_MAE_CANDIDATES = ["model_mae_xt", "forward_restored_expression_mae", "forward_xt_mae"]
    FORWARD_MSE_CANDIDATES = ["model_mse_xt", "forward_restored_expression_mse", "forward_xt_mse"]
    FORWARD_LABEL_PREFIX = "Restored expression"

METRIC_ALIASES = {
    "forward_effect_mae": FORWARD_MAE_CANDIDATES,
    "forward_effect_mse": FORWARD_MSE_CANDIDATES,
    "inverse_accuracy": ["test_accuracy", "inverse_accuracy", "accuracy"],
    "inverse_macro_f1": ["test_macro_f1", "inverse_macro_f1", "macro_f1"],
    "inverse_macro_precision": ["test_macro_precision", "inverse_macro_precision", "macro_precision", "precision"],
    "inverse_macro_recall": ["test_macro_recall", "inverse_macro_recall", "macro_recall", "recall"],
    # Use the AUC value, not test_macro_auc_ovr_n_valid_classes.
    "inverse_macro_auc": ["test_macro_auc_ovr", "inverse_macro_auc", "macro_auc", "test_macro_auc"],
}

METRIC_SOURCE = {
    "forward_effect_mae": "forward",
    "forward_effect_mse": "forward",
    "inverse_accuracy": "inverse",
    "inverse_macro_f1": "inverse",
    "inverse_macro_precision": "inverse",
    "inverse_macro_recall": "inverse",
    "inverse_macro_auc": "inverse",
}

METRIC_LABELS = {
    "forward_effect_mae": f"{FORWARD_LABEL_PREFIX} MAE",
    "forward_effect_mse": f"{FORWARD_LABEL_PREFIX} MSE",
    "inverse_accuracy": "Inverse accuracy",
    "inverse_macro_f1": "Inverse macro F1",
    "inverse_macro_precision": "Inverse macro precision",
    "inverse_macro_recall": "Inverse macro recall",
    "inverse_macro_auc": "Inverse macro AUC",
}

LOWER_IS_BETTER = {
    "forward_effect_mae": True,
    "forward_effect_mse": True,
    "inverse_accuracy": False,
    "inverse_macro_f1": False,
    "inverse_macro_precision": False,
    "inverse_macro_recall": False,
    "inverse_macro_auc": False,
}


# ============================================================
# Helpers to convert summary CSVs into the old plotting table format
# ============================================================

def find_first_existing_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for col in candidates:
        if col in df.columns:
            return col
    return None


def slash_to_slug(x: str) -> str:
    return str(x).strip().strip("/").replace("/", "__")


def slug_to_slash(x: str) -> str:
    return str(x).strip().strip("/").replace("__", "/")


def infer_variant_slug_from_path(value) -> str:
    if pd.isna(value):
        return ""
    p = str(value).replace("\\", "/")

    # Already a strategy slug or strategy path.
    for base in STRATEGY_PLOT_LABELS:
        if p == base or p.startswith(base + "__") or p.startswith(base + "/"):
            return slash_to_slug(p)

    # Pseudo-control h5ad path: .../<PERTURBED_GROUP>/<variant>/pseudo_control_aligned_to_perturbed.h5ad
    suffixes = [
        "/pseudo_control_aligned_to_perturbed.h5ad",
        "/pair_metadata.csv",
    ]
    for suffix in suffixes:
        if p.endswith(suffix):
            parent = p[: -len(suffix)]
            for marker in [f"/{DATASET_ID}/{PERTURBED_GROUP}/", f"/{PERTURBED_GROUP}/"]:
                if marker in parent:
                    return slash_to_slug(parent.split(marker, 1)[1])
            return slash_to_slug(Path(parent).name)

    # Embedding path: .../embeddings/<variant_slug>/X_model.npy
    if "/embeddings/" in p:
        after = p.split("/embeddings/", 1)[1]
        return slash_to_slug(after.split("/", 1)[0])

    # Raw variant_slug or run_id fallback.
    return slash_to_slug(p)


def build_variant_slug(df: pd.DataFrame) -> pd.Series:
    # Prefer columns that directly contain the variant identity.
    for col in ["variant_slug", "pseudo_embedding_slug", "variant_id"]:
        if col in df.columns:
            s = df[col].map(infer_variant_slug_from_path)
            if s.astype(str).str.len().gt(0).any():
                return s

    # Then use paths.
    for col in ["pseudo_control_h5ad", "pseudo_embedding_path", "pair_metadata_path"]:
        if col in df.columns:
            s = df[col].map(infer_variant_slug_from_path)
            if s.astype(str).str.len().gt(0).any():
                return s

    # Last fallback: run_id often starts with strategy name and parameter label.
    if "run_id" in df.columns:
        return df["run_id"].map(infer_variant_slug_from_path)

    raise KeyError(
        "Cannot infer variant_slug. Expected one of: variant_slug, pseudo_embedding_slug, "
        "variant_id, pseudo_control_h5ad, pseudo_embedding_path, pair_metadata_path, run_id."
    )


def get_strategy_base(variant_slug: str) -> str:
    variant_slug = str(variant_slug)
    known_bases = sorted(STRATEGY_PLOT_LABELS.keys(), key=len, reverse=True)
    for base in known_bases:
        if variant_slug == base or variant_slug.startswith(base + "__"):
            return base
    return variant_slug.split("__seed_")[0]


def get_s5_variant_key(variant_slug: str) -> str:
    variant_slug = str(variant_slug)
    if not variant_slug.startswith("S5_SEACell_OT_sampled_average"):
        return ""
    parts = variant_slug.split("__")
    nmc = None
    topk = None
    for part in parts:
        if part.startswith("nmc_"):
            nmc = part.replace("nmc_", "")
        elif part.startswith("topk_"):
            raw = part.replace("topk_", "")
            try:
                topk = str(int(raw))
            except ValueError:
                topk = raw.lstrip("0") or raw
    if nmc is not None and topk is not None:
        return f"{nmc}&{topk}"
    return ""


def get_strategy_label(variant_slug: str) -> str:
    base = get_strategy_base(variant_slug)
    label = STRATEGY_PLOT_LABELS.get(base, str(variant_slug))
    s5_key = get_s5_variant_key(variant_slug)
    if s5_key:
        label = f"{label}\n({s5_key})"
    return label


def get_strategy_color(variant_slug: str) -> str:
    base = get_strategy_base(variant_slug)
    s5_key = get_s5_variant_key(variant_slug)
    if base == "S5_SEACell_OT_sampled_average" and s5_key in S5_VARIANT_COLORS:
        return S5_VARIANT_COLORS[s5_key]
    return STRATEGY_BASE_COLORS.get(base, "#999999")


def get_strategy_color_map(strategy_order: List[str]) -> Dict[str, str]:
    return {s: get_strategy_color(s) for s in strategy_order}


def get_strategy_offsets(strategy_order: List[str], dodge_width: float = DODGE_WIDTH) -> Dict[str, float]:
    n = len(strategy_order)
    if n <= 1 or dodge_width == 0:
        return {s: 0.0 for s in strategy_order}
    offsets = np.linspace(-dodge_width / 2, dodge_width / 2, n)
    return {s: float(o) for s, o in zip(strategy_order, offsets)}


def read_summary_table(path: Path, table_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing {table_name} summary CSV: {path}")
    df = pd.read_csv(path)
    print(f"[Read] {table_name}: {path} | shape={df.shape}")
    return df


def convert_summary_to_plot_records(forward_df: pd.DataFrame, inverse_df: pd.DataFrame) -> pd.DataFrame:
    records = []

    for source_name, df in [("forward", forward_df), ("inverse", inverse_df)]:
        work = df.copy()
        work["model"] = work.get("mlp_representation", work.get("model", "")).astype(str)
        work["variant_slug"] = build_variant_slug(work)
        work["source_table"] = source_name

        # Keep selected strategies only.
        work = work[work["variant_slug"].astype(str).isin(STRATEGY_ORDER)].copy()

        if work.empty:
            print(f"[Warn] No selected variants found in {source_name} summary after filtering.")
            debug_path = OUT_DIR / f"debug_{source_name}_available_variant_slugs.csv"
            tmp = df.copy()
            try:
                tmp["variant_slug_inferred"] = build_variant_slug(tmp)
                tmp[[c for c in ["mlp_representation", "model", "variant_slug", "variant_slug_inferred", "pseudo_control_h5ad", "pseudo_embedding_slug", "pseudo_embedding_path", "run_id"] if c in tmp.columns]].drop_duplicates().to_csv(debug_path, index=False)
                print(f"[Debug] Wrote available variants: {debug_path}")
            except Exception:
                pass
            continue

        for metric in METRICS_TO_PLOT:
            if METRIC_SOURCE[metric] != source_name:
                continue
            src_col = find_first_existing_column(work, METRIC_ALIASES[metric])
            if src_col is None:
                print(f"[Skip] {metric}: none of {METRIC_ALIASES[metric]} found in {source_name} summary.")
                continue
            work[metric] = pd.to_numeric(work[src_col], errors="coerce")

        keep_cols = ["model", "variant_slug"] + [m for m in METRICS_TO_PLOT if m in work.columns]
        records.append(work[keep_cols])

    if not records:
        raise RuntimeError("No selected variant records were found in the forward/inverse summary CSVs.")

    out = pd.concat(records, ignore_index=True, sort=False)

    # Merge forward and inverse metrics into one row per model/variant.
    metric_cols = [m for m in METRICS_TO_PLOT if m in out.columns]
    out = (
        out.groupby(["model", "variant_slug"], dropna=False)[metric_cols]
        .mean(numeric_only=True)
        .reset_index()
    )

    return out


def prepare_plot_df(results_df: pd.DataFrame) -> pd.DataFrame:
    df = results_df.copy()

    required = ["model", "variant_slug"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    df = df[df["model"].astype(str).isin(MODELS_ORDER)].copy()
    df["model"] = pd.Categorical(df["model"].astype(str), categories=MODELS_ORDER, ordered=True)

    observed_strategies = list(df["variant_slug"].dropna().astype(str).unique())
    strategy_order = [s for s in STRATEGY_ORDER if s in observed_strategies]
    strategy_order += [s for s in observed_strategies if s not in strategy_order]
    df["variant_slug"] = pd.Categorical(df["variant_slug"].astype(str), categories=strategy_order, ordered=True)

    df["strategy_base"] = df["variant_slug"].astype(str).map(get_strategy_base)
    df["s5_variant_key"] = df["variant_slug"].astype(str).map(get_s5_variant_key)
    df["strategy_label"] = df["variant_slug"].astype(str).map(get_strategy_label)
    df["strategy_color"] = df["variant_slug"].astype(str).map(get_strategy_color)

    df = df.sort_values(["model", "variant_slug"]).reset_index(drop=True)
    return df


# ============================================================
# Plotting: same layout as Visualize_metrics.ipynb, but no connecting lines
# ============================================================

def plot_metric_scatter(
    df: pd.DataFrame,
    metric: str,
    out_dir: Path,
    models_order: List[str] = MODELS_ORDER,
    point_size: int = POINT_SIZE,
    dodge_width: float = DODGE_WIDTH,
):
    if metric not in df.columns:
        print(f"[Skip] metric not found: {metric}")
        return None

    mdf = df.dropna(subset=[metric]).copy()
    if mdf.empty:
        print(f"[Skip] no valid values for metric: {metric}")
        return None

    strategy_order = list(mdf["variant_slug"].cat.categories)
    strategy_order = [s for s in strategy_order if s in set(mdf["variant_slug"].astype(str))]
    color_map = get_strategy_color_map(strategy_order)
    offset_map = get_strategy_offsets(strategy_order, dodge_width=dodge_width)

    models_present = [m for m in models_order if m in set(mdf["model"].astype(str))]
    model_x = {m: i for i, m in enumerate(models_present)}

    fig, ax = plt.subplots(figsize=FIGSIZE)

    for strategy in strategy_order:
        sdf = mdf[mdf["variant_slug"].astype(str).eq(strategy)].copy()
        if sdf.empty:
            continue

        xs = []
        ys = []
        for _, row in sdf.iterrows():
            model = str(row["model"])
            if model not in model_x:
                continue
            xs.append(model_x[model] + offset_map[strategy])
            ys.append(float(row[metric]))

        if not xs:
            continue

        ax.scatter(
            xs,
            ys,
            s=point_size,
            alpha=ALPHA,
            label=get_strategy_label(strategy).replace("\n", " "),
            color=color_map[strategy],
            edgecolor="black",
            linewidth=0.4,
            zorder=4,
        )

    # Important: no ax.plot() here. This removes the lines linking strategies across models.

    ax.set_xticks(range(len(models_present)))
    ax.set_xticklabels(models_present, rotation=0)
    ax.set_xlabel("Embedding / input model")
    ax.set_ylabel(METRIC_LABELS.get(metric, metric))
    title_suffix = "lower is better" if LOWER_IS_BETTER.get(metric, False) else "higher is better"
    ax.set_title(f"{METRIC_LABELS.get(metric, metric)} by model and pseudo-control strategy ({title_suffix})")

    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.45)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(
        title="Pseudo-control strategy",
        bbox_to_anchor=(1.02, 1.0),
        loc="upper left",
        borderaxespad=0,
        frameon=False,
        fontsize=8,
        title_fontsize=9,
    )

    fig.tight_layout()

    png_path = out_dir / f"{metric}_strategy_scatter_no_lines.png"
    pdf_path = out_dir / f"{metric}_strategy_scatter_no_lines.pdf"
    svg_path = out_dir / f"{metric}_strategy_scatter_no_lines.svg"

    if SAVE_PNG:
        fig.savefig(png_path, dpi=DPI, bbox_inches="tight")
    if SAVE_PDF:
        fig.savefig(pdf_path, bbox_inches="tight")
    if SAVE_SVG:
        fig.savefig(svg_path, bbox_inches="tight")

    if DISPLAY_FIGURES and display is not None:
        display(fig)
    else:
        plt.show()
    plt.close(fig)

    print(f"[Plot] Saved: {png_path if SAVE_PNG else ''}")
    if SAVE_PDF:
        print(f"[Plot] Saved: {pdf_path}")
    if SAVE_SVG:
        print(f"[Plot] Saved: {svg_path}")
    return png_path


def summarize_best_strategy(df: pd.DataFrame, metrics: List[str]) -> pd.DataFrame:
    rows = []
    for metric in metrics:
        if metric not in df.columns:
            continue

        lower = LOWER_IS_BETTER.get(metric, False)
        for model in MODELS_ORDER:
            sub = df[df["model"].astype(str).eq(model)].dropna(subset=[metric]).copy()
            if sub.empty:
                continue

            best = sub.loc[sub[metric].idxmin()] if lower else sub.loc[sub[metric].idxmax()]

            rows.append(
                {
                    "metric": metric,
                    "source_column_mode": FORWARD_ERROR_MODE if metric.startswith("forward") else "inverse",
                    "direction": "lower_is_better" if lower else "higher_is_better",
                    "model": model,
                    "best_variant_slug": str(best["variant_slug"]),
                    "best_strategy_label": get_strategy_label(str(best["variant_slug"])).replace("\n", " "),
                    "best_value": float(best[metric]),
                }
            )

    return pd.DataFrame(rows)


def main():
    forward_df = read_summary_table(FORWARD_SUMMARY_CSV, "forward")
    inverse_df = read_summary_table(INVERSE_SUMMARY_CSV, "inverse")

    results_df = convert_summary_to_plot_records(forward_df, inverse_df)
    scan_path = OUT_DIR / "summary_csv_scan_selected_records.csv"
    results_df.to_csv(scan_path, index=False)
    print(f"[Scan] Saved selected summary scan: {scan_path}")

    plot_df = prepare_plot_df(results_df)
    plot_table_path = OUT_DIR / "plotting_table.csv"
    plot_df.to_csv(plot_table_path, index=False)
    print(f"[Table] Saved plotting table: {plot_table_path}")

    saved_plots = []
    for metric in METRICS_TO_PLOT:
        p = plot_metric_scatter(plot_df, metric, OUT_DIR)
        if p is not None:
            saved_plots.append(str(p))

    best_df = summarize_best_strategy(plot_df, METRICS_TO_PLOT)
    best_path = OUT_DIR / "best_strategy_by_model_metric.csv"
    best_df.to_csv(best_path, index=False)
    print(f"[Best] Saved: {best_path}")

    print("Saved outputs:")
    print(f"  Selected summary scan: {scan_path}")
    print(f"  Plotting table:        {plot_table_path}")
    print(f"  Best table:            {best_path}")
    print(f"  Figures:               {OUT_DIR}")
    return saved_plots


if __name__ == "__main__":
    main()
